In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import xarray as xr
import zarr
import icechunk
import fsspec
from icechunk.xarray import to_icechunk
from dask.distributed import Client
from evaltools.source import get_source_collection, open_and_sort, open_datasets

In [3]:
client = Client(dashboard_address="localhost:8787", threads_per_worker=1)

In [4]:
catalog = get_source_collection(project_id="CORDEX-CMIP6", variable_id=["tas", "pr"], frequency="mon", add_fx=["sftlf", "orog"], driving_experiment_id="evaluation")
dsets = open_and_sort(catalog, merge_fx=True)

Opening catalog from https://raw.githubusercontent.com/euro-cordex/joint-evaluation/refs/heads/main/CORDEX-CMIP6.json
Found 15 datasets for variables ['tas', 'pr']: ['RACMO23E', 'COSMO-CLM-6-0-clm3', 'ICON-CLM-202407-1-1', 'CCLM6-0-1-URB', 'HCLIM43-ALADIN', 'ROAM-NBS', 'REMO2020-2-2-iMOVE-LUC', 'REMO2020-2-2-MR2', 'REMO2020-2-2', 'REMO2020-2-2-iMOVE', 'WRF451Q', 'CCLM6-0-1-URB-ESG', 'CNRM-ALADIN64E1', 'RegCM5-0', 'ALARO1-SFX']

--> The keys in the returned dictionary of datasets are constructed as follows:
	'project_id.domain_id.institution_id.driving_source_id.driving_experiment_id.driving_variant_label.source_id.version_realization.frequency.version'


2025-10-11 21:18:25,977 - distributed.worker - ERROR - Compute Failed
Key:       _delayed_open_ds-fc3c221a-b907-45cb-ae64-208fa9fc5437
State:     executing
Task:  <Task '_delayed_open_ds-fc3c221a-b907-45cb-ae64-208fa9fc5437' _delayed_open_ds(..., ...)>
Exception: 'ValueError(\'Failed to decode variable \\\'time\\\': unable to decode time units \\\'months since 1980-1-1 00:00:00\\\' with "calendar \\\'standard\\\'". Try opening your dataset with decode_times=False or installing cftime if it is not installed.\')'
Traceback: '  File "/mnt/CORDEX_CMIP6_tmp/user_tmp/lbuntemeyer/conda_envs/cordex-etl/lib/python3.13/site-packages/intake_esm/source.py", line 60, in _delayed_open_ds\n    return _open_dataset(*args, **kwargs)\n  File "/mnt/CORDEX_CMIP6_tmp/user_tmp/lbuntemeyer/conda_envs/cordex-etl/lib/python3.13/site-packages/intake_esm/source.py", line 99, in _open_dataset\n    ds = xr.open_dataset(url, **xarray_open_kwargs)\n  File "/mnt/CORDEX_CMIP6_tmp/user_tmp/lbuntemeyer/conda_envs/cordex

decoding dataset CORDEX-CMIP6.EUR-12.CLMcom-CMCC.ERA5.evaluation.r1i1p1f1.CCLM6-0-1-URB.v1-r1.fx.v20250201
Found 30 datasets
decoding dataset CORDEX-CMIP6.EUR-12.KNMI.ERA5.evaluation.r1i1p1f1.RACMO23E.v1-r1.fx.v20241216
Found 30 datasets
decoding dataset CORDEX-CMIP6.EUR-12.ICTP.ERA5.evaluation.r1i1p1f1.RegCM5-0.v1-r1.fx.v20250415
Found 30 datasets
decoding dataset CORDEX-CMIP6.EUR-12.CLMcom-Hereon.ERA5.evaluation.r1i1p1f1.ICON-CLM-202407-1-1.v1-r1.fx.v20240920
Warning for CORDEX-CMIP6.EUR-12.CLMcom-Hereon.ERA5.evaluation.r1i1p1f1.ICON-CLM-202407-1-1.v1-r1.fx.v20240920: Variable(s) referenced in cell_measures not in variables: ['areacella']
Warning for CORDEX-CMIP6.EUR-12.CLMcom-Hereon.ERA5.evaluation.r1i1p1f1.ICON-CLM-202407-1-1.v1-r1.fx.v20240920: Variable(s) referenced in cell_measures not in variables: ['areacella']
Found 30 datasets
decoding dataset CORDEX-CMIP6.EUR-12.RMIB-UGent.ERA5.evaluation.r1i1p1f1.ALARO1-SFX.v1-r1.fx.v20241009
Found 30 datasets
decoding dataset CORDEX-CMIP6

## Write to S3 without icechunk

In [5]:
def rechunk(ds, refvar=None):
    """Naive rechunking along time"""
    if refvar is None:
        refvar = [var for var in ds.data_vars if "time" in ds[var].dims][0]
    chunksize = int(ds[refvar].sizes["time"] * 100 * 1024**2 / ds[refvar].nbytes)
    chunks = {dim: chunksize if dim == "time" else -1 for dim in ds.dims}
    print(f"Rechunking to {chunks}")
    return ds.chunk(chunks)

def write_to_s3(key, ds):
    target = fsspec.get_mapper(f"s3://euro-cordex/CORDEX-CMIP6/{key}")
    rechunk(ds).to_zarr(target, zarr_format=3, compute=True)

for key, ds in dsets.items():
    print(f"Writing {key} to s3...")
    write_to_s3(key, ds)

Writing CORDEX-CMIP6.EUR-12.KNMI.ERA5.evaluation.r1i1p1f1.RACMO23E.v1-r1.mon.v20241216 to s3...
Rechunking to {'time': 150, 'rlat': -1, 'rlon': -1, 'bnds': -1}


/mnt/CORDEX_CMIP6_tmp/user_tmp/lbuntemeyer/conda_envs/cordex-etl/lib/python3.13/site-packages/zarr/core/dtype/npy/bytes.py:383: UnstableSpecificationWarning: The data type (NullTerminatedBytes(length=1)) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/mnt/CORDEX_CMIP6_tmp/user_tmp/lbuntemeyer/conda_envs/cordex-etl/lib/python3.13/site-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Writing CORDEX-CMIP6.EUR-12.GERICS.ERA5.evaluation.r1i1p1f1.REMO2020-2-2.v1-r1.mon.v20241120 to s3...
Rechunking to {'time': 150, 'rlat': -1, 'rlon': -1, 'bnds': -1, 'vertices': -1}


/mnt/CORDEX_CMIP6_tmp/user_tmp/lbuntemeyer/conda_envs/cordex-etl/lib/python3.13/site-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Writing CORDEX-CMIP6.EUR-12.GERICS.ERA5.evaluation.r1i1p1f1.REMO2020-2-2-iMOVE-LUC.v1-r1.mon.v20250515 to s3...
Rechunking to {'time': 150, 'rlat': -1, 'rlon': -1, 'bnds': -1, 'vertices': -1}


...

/mnt/CORDEX_CMIP6_tmp/user_tmp/lbuntemeyer/conda_envs/cordex-etl/lib/python3.13/site-packages/zarr/core/dtype/npy/bytes.py:383: UnstableSpecificationWarning: The data type (NullTerminatedBytes(length=1)) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/mnt/CORDEX_CMIP6_tmp/user_tmp/lbuntemeyer/conda_envs/cordex-etl/lib/python3.13/site-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Writing CORDEX-CMIP6.EUR-12.CLMcom-Hereon.ERA5.evaluation.r1i1p1f1.ICON-CLM-202407-1-1.v1-r1.mon.v20240920 to s3...
Rechunking to {'time': 150, 'rlat': -1, 'rlon': -1, 'bnds': -1, 'vertices': -1}


/mnt/CORDEX_CMIP6_tmp/user_tmp/lbuntemeyer/conda_envs/cordex-etl/lib/python3.13/site-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Writing CORDEX-CMIP6.EUR-12.CNRM-MF.ERA5.evaluation.r1i1p1f1.CNRM-ALADIN64E1.v1-r1.mon.v20250505 to s3...
Rechunking to {'time': 127, 'y': -1, 'x': -1, 'vertices': -1, 'bnds': -1}


/mnt/CORDEX_CMIP6_tmp/user_tmp/lbuntemeyer/conda_envs/cordex-etl/lib/python3.13/site-packages/zarr/core/dtype/npy/bytes.py:383: UnstableSpecificationWarning: The data type (NullTerminatedBytes(length=255)) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/mnt/CORDEX_CMIP6_tmp/user_tmp/lbuntemeyer/conda_envs/cordex-etl/lib/python3.13/site-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


## Open with xarray and obstore

In [15]:
import xarray as xr
from zarr.storage import ObjectStore
from obstore.store import S3Store

url = "s3://euro-cordex/CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.GERICS.ERA5.evaluation.r1i1p1f1.REMO2020-2-2.v1-r1.mon.v20241120"
prefix="CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.GERICS.ERA5.evaluation.r1i1p1f1.REMO2020-2-2.v1-r1.mon.v20241120"
s3_store = S3Store('euro-cordex', skip_signature=True, region="eu-central-1", prefix=prefix)
store = ObjectStore(s3_store, read_only=True)
ds = xr.open_dataset(store, consolidated=True, engine="zarr")
ds

<xarray.Dataset> Size: 703MB
Dimensions:                     (rlat: 412, rlon: 424, time: 492, bnds: 2,
                                 vertices: 4)
Coordinates:
    height                      float64 8B ...
    lat                         (rlat, rlon) float64 1MB ...
    lon                         (rlat, rlon) float64 1MB ...
  * rlat                        (rlat) float64 3kB -23.38 -23.27 ... 21.73 21.84
  * rlon                        (rlon) float64 3kB -28.38 -28.27 ... 18.05 18.16
  * time                        (time) datetime64[ns] 4kB 1980-01-16T12:00:00...
Dimensions without coordinates: bnds, vertices
Data variables:
    orog                        (rlat, rlon) float32 699kB ...
    pr                          (time, rlat, rlon) float32 344MB ...
    rotated_latitude_longitude  int32 4B ...
    sftlf                       (rlat, rlon) float32 699kB ...
    tas                         (time, rlat, rlon) float32 344MB ...
    time_bnds                   (time, bnds) datetime64[ns] 8kB ...
    vertices_lat                (rlat, rlon, vertices) float64 6MB ...
    vertices_lon                (rlat, rlon, vertices) float64 6MB ...
Attributes: (12/56)
    CORDEX_domain:                           EUR-11
    Conventions:                             CF-1.11
    activity_id:                             DD
    comment:                                 original REMO run id: 065020
    contact:                                 gerics-cordex@hereon.de
    creation_date:                           2025-09-25T16:13:15Z
    ...                                      ...
    intake_esm_attrs:version:                v20241120
    intake_esm_attrs:time_range:             197901-198812
    intake_esm_attrs:variable_id:            tas
    intake_esm_attrs:path:                   /mnt/CORDEX_CMIP6_tmp/sim_data/C...
    intake_esm_attrs:_data_format_:          netcdf
    intake_esm_dataset_key:                  CORDEX-CMIP6.EUR-12.GERICS.ERA5....

## Virtualization with virtualizarr

In [ ]:
from virtualizarr import open_virtual_dataset
from virtualizarr.registry import ObjectStoreRegistry
from virtualizarr.parsers import ZarrParser
from obstore.store import from_url
from obstore.store import S3Store

import nest_asyncio
nest_asyncio.apply()

bucket = "euro-cordex"
# Dataset prefix (root of the Zarr store). Must match where zarr.json lives.
prefix = "CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.GERICS.ERA5.evaluation.r1i1p1f1.REMO2020-2-2.v1-r1.mon.v20241120"
root_url = f"s3://{bucket}/{prefix}"

# IMPORTANT: include prefix so the store's root IS the Zarr group root
s3_store = S3Store(bucket, prefix=prefix, skip_signature=True, region="eu-central-1")

from zarr.storage import ObjectStore
store = ObjectStore(s3_store, read_only=True)

registry = ObjectStoreRegistry({root_url: s3_store})
parser = ZarrParser(root_url)  # NOTE: current virtualizarr may only support Zarr v2 groups

try:
    vds = open_virtual_dataset(
        root_url,
        parser=parser,
        registry=registry,
    )
except Exception as e:
    print("open_virtual_dataset failed:", repr(e))
    vds = None

if vds is not None:
    print("Variables in virtual dataset:", list(vds.data_vars))
    if "tas" in vds:
        print("tas shape:", vds.tas.shape)
    display(vds)
else:
    print("Will run diagnostic cell next to inspect Zarr format (v2 vs v3)")

GroupNotFoundError: No group found in store ObjectStore(object_store://S3Store(bucket="euro-cordex", prefix="CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.GERICS.ERA5.evaluation.r1i1p1f1.REMO2020-2-2.v1-r1.mon.v20241120")) at path 's3:/euro-cordex/CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.GERICS.ERA5.evaluation.r1i1p1f1.REMO2020-2-2.v1-r1.mon.v20241120'

In [18]:
import fsspec
fs = fsspec.filesystem("s3", anon=True)
fs.ls(f"euro-cordex/{prefix}")#[:10]

['euro-cordex/CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.GERICS.ERA5.evaluation.r1i1p1f1.REMO2020-2-2.v1-r1.mon.v20241120/height',
 'euro-cordex/CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.GERICS.ERA5.evaluation.r1i1p1f1.REMO2020-2-2.v1-r1.mon.v20241120/lat',
 'euro-cordex/CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.GERICS.ERA5.evaluation.r1i1p1f1.REMO2020-2-2.v1-r1.mon.v20241120/lon',
 'euro-cordex/CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.GERICS.ERA5.evaluation.r1i1p1f1.REMO2020-2-2.v1-r1.mon.v20241120/orog',
 'euro-cordex/CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.GERICS.ERA5.evaluation.r1i1p1f1.REMO2020-2-2.v1-r1.mon.v20241120/pr',
 'euro-cordex/CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.GERICS.ERA5.evaluation.r1i1p1f1.REMO2020-2-2.v1-r1.mon.v20241120/rlat',
 'euro-cordex/CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.GERICS.ERA5.evaluation.r1i1p1f1.REMO2020-2-2.v1-r1.mon.v20241120/rlon',
 'euro-cordex/CORDEX-CMIP6/CORDEX-CMIP6.EUR-12.GERICS.ERA5.evaluation.r1i1p1f1.REMO2020-2-2.v1-r1.mon.v20241120/rotated_latitude_longitude',
 'euro-cordex/CORDEX-CMIP6/C

In [ ]:
# Diagnostics: inspect the Zarr store to determine version and root keys
import json, io, fsspec, zarr
from zarr.storage import ObjectStore

fs = fsspec.filesystem("s3", anon=True)
keys = fs.ls(f"euro-cordex/{prefix}")
print("Sample keys:")
for k in keys[:20]:
    print(" ", k)

# Check for v3 vs v2 indicators
has_zarr_json = fs.exists(f"euro-cordex/{prefix}/zarr.json")
has_zgroup = fs.exists(f"euro-cordex/{prefix}/.zgroup")
print(f"zarr.json present: {has_zarr_json}; .zgroup present: {has_zgroup}")

if has_zarr_json:
    text = fs.open(f"euro-cordex/{prefix}/zarr.json").read().decode()
    meta = json.loads(text)
    print("Detected zarr_format in zarr.json:", meta.get("zarr_format"))

# Attempt manual open using zarr (v3) API if available
try:
    import zarr
    grp = zarr.open_group(store, mode="r")  # will infer format
    print("Opened group via zarr.open_group; arrays:", list(grp.array_keys()))
except Exception as e:
    print("zarr.open_group failed:", e)

# If library limitation (virtualizarr expects v2), propose workaround
if has_zarr_json and not has_zgroup:
    print("This is a Zarr v3 store. virtualizarr version installed may only support v2; need upgrade or conversion.")

## Create icechunk repository

In [ ]:
# create on store per dataset

def create_icechunk_repo(key, ds):
    print(f"Adding {key} to icechunk...")
    storage_config = icechunk.s3_storage(
        bucket="euro-cordex",
        prefix=f"CORDEX-CMIP6_icechunk/{key}",
        region='eu-central-1',
    )
    repo = icechunk.Repository.create(storage_config)
    session = repo.writable_session("main")
    to_icechunk(ds, session)
    first_snapshot = session.commit("initial")
    print(f"Finished {key}: {first_snapshot}, {session.status}")

for key, ds in dsets.items():
    create_icechunk_repo(key, ds)

In [ ]:
# create one store with one dataset per group

def create_icechunk_repo_all_in_one(dsets):
    storage_config = icechunk.s3_storage(
        bucket="euro-cordex",
        prefix=f"CORDEX-CMIP6_icechunk_groups",
        region='eu-central-1',
    )
    repo = icechunk.Repository.create(storage_config)
    session = repo.writable_session("main")
    for key, ds in dsets.items():
        print(f"Adding {key} to icechunk...")
        #group = zarr.create_group(session.store, path=key, zarr_format=3)
        ds.load().to_zarr(session.store, group=key, zarr_format=3, consolidated=False)
        #to_icechunk(ds, session, group=key)
    first_snapshot = session.commit("initial")
    print(f"Finished {key}: {first_snapshot}, {session.status}")

create_icechunk_repo_all_in_one(dsets)  # only tas and pr for testing

In [ ]:
storage = icechunk.s3_storage(
    bucket="euro-cordex",
    prefix=f"CORDEX-CMIP6-groups",
    region='eu-central-1',
    )
repo = icechunk.Repository.open(storage)
session = repo.readonly_session("main")
dt = xr.open_datatree(session.store, consolidated=False, engine="zarr")

In [ ]:
# open all single stores again

for prefix in dsets.keys():
    storage = icechunk.s3_storage(bucket="euro-cordex", prefix=prefix, region="eu-central-1")
    repo = icechunk.Repository.open(storage)
    session = repo.readonly_session("main")
    ds = xr.open_zarr(session.store, zarr_format=3, consolidated=False)


In [ ]:
hist = repo.ancestry(branch="main")
for ancestor in hist:
    print(ancestor.id, ancestor.message, ancestor.written_at)

In [ ]:
session = repo.readonly_session("main")
ds = xr.open_zarr(session.store, zarr_format=3, consolidated=False)

In [ ]:
ds